# China Tier-3A International Medical Centers & Cross-Border Medical Cost Benchmarks (2026 EDA)

Reproducible exploratory analysis of the **ChinaMedVisit Open Healthcare Benchmark Dataset (CC-BY-4.0)**:
- **76 verified hospitals across 10 Chinese cities** (74 public Grade-3A IMCs + 2 JCI private international hospitals)
- **12 core clinical procedures** comparing China Tier-3A IMC out-of-pocket ranges against Chinese private clinics, US self-pay averages, and UK private rates (reference rate: `0.138 USD/RMB`)
- **4 ancillary coordination items + 4 bundled multi-day care presets**
- **24 frontier specialty therapies** across 8 clinical specialties & **12 deep-dive clinical comparison rows**

**Live Interactive Tools & Citation**:
- Cost Calculator: [chinamedvisit.com/cost-calculator](https://chinamedvisit.com/cost-calculator)
- Verified Hospital Directory: [chinamedvisit.com/hospitals](https://chinamedvisit.com/hospitals)
- License: `CC-BY-4.0` (data) / `MIT` (notebook code)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve dataset folder seamlessly in both local repo and Kaggle kernels
candidates = [Path("../data"), Path("./data"), Path("/kaggle/input")]
BASE = None
for c in candidates:
    if c.exists():
        hits = list(c.rglob("china_tier3a_imc_hospitals_76.csv"))
        if hits:
            BASE = hits[0].parent
            break
assert BASE is not None, "Dataset CSV files not found."

hospitals = pd.read_csv(BASE / "china_tier3a_imc_hospitals_76.csv")
costs = pd.read_csv(BASE / "china_vs_us_uk_medical_costs_2026.csv")
ancillary = pd.read_csv(BASE / "china_medical_travel_ancillary_and_packages_2026.csv")
therapies = pd.read_csv(BASE / "china_specialty_frontier_therapies_24.csv")
deepdive = pd.read_csv(BASE / "china_specialty_deepdive_comparisons_12.csv")

print(f"Hospitals: {len(hospitals)} across {hospitals['city'].nunique()} cities")
print(f"Core Procedures: {len(costs)} | Ancillary & Packages: {len(ancillary)}")
print(f"Frontier Therapies: {len(therapies)} across {therapies['specialty_slug'].nunique()} specialties | Deep-Dive Rows: {len(deepdive)}")
display(costs[["procedure_id", "tier3a_imc_rmb_min", "tier3a_imc_rmb_max", "tier3a_imc_usd_min", "tier3a_imc_usd_max", "us_average_usd", "uk_private_gbp", "estimated_savings_vs_us_pct"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

costs["imc_usd_mid"] = (costs["tier3a_imc_usd_min"] + costs["tier3a_imc_usd_max"]) / 2
plot_df = costs.sort_values("us_average_usd", ascending=True)
labels = plot_df["procedure_id"].str.replace("-", " ").str.title()
y = list(range(len(plot_df)))
axes[0].barh([i - 0.18 for i in y], plot_df["imc_usd_mid"], height=0.35, label="China Tier-3A IMC (USD Midpoint)", color="#1b4332")
axes[0].barh([i + 0.18 for i in y], plot_df["us_average_usd"], height=0.35, label="US Self-Pay Average (USD)", color="#64748b")
axes[0].set_yticks(y)
axes[0].set_yticklabels(labels, fontsize=9)
axes[0].set_xlabel("Estimated Out-of-Pocket Cost (USD)")
axes[0].set_title("China Tier-3A IMC vs US Self-Pay Benchmark (12 Core Procedures)")
axes[0].legend()

city_counts = hospitals["city"].value_counts().sort_values()
city_counts.plot(kind="barh", ax=axes[1], color="#2d6a4f")
axes[1].set_title("76 Verified Hospitals Across 10 Chinese Cities")
axes[1].set_xlabel("Verified Hospital Count")

plt.tight_layout()
plt.show()


In [ ]:
# Direct-billing international insurer coverage frequency across all 76 hospitals
insurer_series = (
    hospitals["direct_billing_insurers"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
)
insurer_series = insurer_series[insurer_series != ""]
print("Top 10 Direct-Billing Insurer Partnerships Listed Across 76 Hospitals:")
display(insurer_series.value_counts().head(10))


## Key Empirical Takeaways
1. **76% to 90% Lower Out-of-Pocket Costs vs US Self-Pay**: Across all 12 core clinical procedures (e.g., 3.0T Brain MRI at ¥1,200–¥2,400 / $166–$331 vs $2,200 US; whole-body 18F-FDG PET-CT at ¥7,000–¥11,000 / $966–$1,518 vs $5,800 US), public Tier-3A International Medical Centers maintain substantial regulated cost advantages.
2. **Geographic Concentration**: Shenzhen (16), Shanghai (13), Guangzhou (11), and Beijing (7) anchor international hospital capacity, complemented by regional university referral hubs in Chengdu, Hangzhou, Wuhan, Xi'an, Haikou, and Suzhou.

Run custom multi-procedure trip bundles interactively at [https://chinamedvisit.com/cost-calculator](https://chinamedvisit.com/cost-calculator).
